# Day I6 — 에너지와 동역학적 보행: 역진자, 충돌, 그리고 걷기의 비용

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I6.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 역진자 한 step: PE와 KE의 교환

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

g = 9.81

def stance(L=0.9, step=0.7, v_mid=1.2, m=70.0):
    """질량 m이 길이 L의 가벼운 다리 끝에 얹힌 역진자.
    θ = 수직선에서 앞으로 기운 각(rad). 운동방정식 θ'' = (g/L) sin θ.
    한 step 동안 θ는 −α → +α (α = asin(step/2 / L)). 중간(θ = 0, 가장 높고 가장 느린 점)의 속력이 v_mid가 되도록 시작 속도를 정한다."""
    a = np.arcsin(step / 2 / L)
    w0 = np.sqrt((v_mid / L) ** 2 + 2 * g / L * (1 - np.cos(a)))   # 중간(가장 높은 점)에서 v_mid가 되도록 에너지 보존으로 역산
    hit = lambda t, s: s[0] - a                                      # θ가 +α에 닿으면 멈춤
    hit.terminal = True
    sol = solve_ivp(lambda t, s: [s[1], g / L * np.sin(s[0])], [0, 3], [-a, w0],
                    events=hit, max_step=1e-3, rtol=1e-9)
    th, w = sol.y
    y = L * np.cos(th)                            # COM 높이
    v = L * w                                     # COM 속력(원 궤도 위)
    PE = m * g * (y - y.min())                    # 가장 낮은 점을 0으로
    KE = 0.5 * m * v ** 2
    return sol.t, th, y, v, PE, KE, a

t, th, y, v, PE, KE, a = stance()
E = PE + KE
print("step 시간 %.3f s, COM 높이 변화 %.1f cm, 속력 %.2f → %.2f → %.2f m/s"
      % (t[-1], 100 * (y.max() - y.min()), v[0], v[np.argmax(y)], v[-1]))
print("PE 최대 시점 = KE 최소 시점? %s | 총에너지 변동 %.2e J (보존계라 거의 0)"
      % (np.argmax(PE) == np.argmin(KE), E.max() - E.min()))
r = (np.abs(np.diff(PE)).sum() + np.abs(np.diff(KE)).sum() - np.abs(np.diff(E)).sum()) / \
    (np.abs(np.diff(PE)).sum() + np.abs(np.diff(KE)).sum())
print("PE–KE 교환 비율(recovery) = %.0f%%" % (100 * r))

fig, ax = plt.subplots(figsize=(6, 3.4))
pct = 100 * t / t[-1]
ax.plot(pct, PE, label="potential (PE)")
ax.plot(pct, KE - KE.min(), label="kinetic (KE − min)")
ax.plot(pct, E - E.min(), "k--", label="total (− min)")
ax.set_xlabel("% of single stance (inverted pendulum)"); ax.set_ylabel("energy change (J)")
ax.legend(); ax.set_title("simulated inverted-pendulum stance, L = 0.9 m")
out = Path("figures/advanced/I/I6_pe_ke.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=150)
print("저장:", out)

print("\n다리 길이·속도를 바꾸면 (step 0.7 m)")
print(" L(m)  v_mid(m/s)  step 시간(s)  COM 높이 변화(cm)  v²/(gL)")
for L in (0.8, 0.9, 1.0):
    for vm in (1.0, 1.4):
        t, th, y, v, PE, KE, a = stance(L=L, v_mid=vm)
        print(" %.1f     %.1f        %.3f          %.1f            %.2f"
              % (L, vm, t[-1], 100 * (y.max() - y.min()), vm ** 2 / (g * L)))

### 예시 2 — 충돌 손실은 보폭의 4제곱으로 커진다

In [ ]:
import numpy as np

# 가장 단순한 충돌 모델: heel strike 순간 COM 속도 방향이 두 다리 사이 각 2α만큼 꺾이고,
# 새 다리 방향(다리를 따라 미는 충격) 성분은 사라진다 → 남는 속력 = v·cos(2α)
L, m, f = 0.9, 70.0, 1.8            # 다리 길이(m), 질량(kg), step 빈도(steps/s)를 고정
print("step(m)  α(deg)  속도(m/s)  KE 중 잃는 비율  step당 손실(J)  손실/step⁴")
for s in [0.5, 0.6, 0.7, 0.8]:
    a = np.arcsin(s / 2 / L)
    v = s * f                         # 빈도를 고정하면 속도는 step 길이에 비례(충돌 직전 속력을 평균 속도로 근사)
    lost_frac = 1 - np.cos(2 * a) ** 2
    lost = 0.5 * m * v ** 2 * lost_frac
    print(" %.1f     %5.1f    %.2f       %5.1f%%          %5.1f        %.0f"
          % (s, np.degrees(a), v, 100 * lost_frac, lost, lost / s ** 4))

### 예시 3 — 시간당 비용과 거리당 비용

In [ ]:
import numpy as np

# [W] 2장 p.37의 두 경험식(체중 1 kg당): 시간당 Ew = 2.23 + 1.26 v² (W/kg), 거리당 Em = Ew / v (J/m/kg)
v = np.linspace(0.5, 2.0, 151)
Ew = 2.23 + 1.26 * v ** 2
Em = Ew / v
i = np.argmin(Em)
print("거리당 비용이 가장 낮은 속도 = %.2f m/s (해석해 √(2.23/1.26) = %.2f)" % (v[i], np.sqrt(2.23 / 1.26)))
for vv in (0.6, 1.0, 1.33, 1.8):
    print("v = %.2f m/s: 시간당 %.2f W/kg, 거리당 %.2f J/m/kg" % (vv, 2.23 + 1.26 * vv ** 2, (2.23 + 1.26 * vv ** 2) / vv))

# [P] 24장의 성인 식: O2 rate(mL/kg·min) = 0.129·V + 2.60 (V: m/min), cost = rate / V
V = 82
rate = 0.129 * V + 2.60
print("\n정상 성인 82 m/min: O2 rate %.1f mL/kg·min, O2 cost %.3f mL/kg·m" % (rate, rate / V))
V_pt = 40                                       # 가상 환자: 같은 rate를 유지하려고 40 m/min으로 걷는다
print("가상 환자 40 m/min, rate를 %.1f로 유지하면 cost = %.3f mL/kg·m (정상의 %.1f배)"
      % (rate, rate / V_pt, (rate / V_pt) / (rate / V)))